# **Overview of dataset.**

Three **progressive messy datasets** designed to simulate real-world data cleaning scenarios that analysts encounter dail. Each dataset increases in complexity and introduces realistic data quality issues.

## **LEVEL 1: Easy-E-commerce Orders Dataset.**<br>
**Business Context**

E-commerce platform order management system with customer transaction data spanning 2023.

### Column Descriptions

1. **orderID** - Unique order identifier (ORD000001, ORD000002, etc.)
2. **CustomerID** - Customer refrences (CUST00001-CUST00500)
3. **ProductName** - Product purchased
4. **Category** - Product category (Electronics, Accessories, Peripherals)
5. **Quantity** - Units ordered (1-20)
6. **UnitPrice** - Price per unit ($10-$2000)
7. **OrderDate** - When order was placed (mixed formats)
8. **ShipDate** - When order was shipped (mixed formats)
9. **DeliveryDate** - When order was delivered
10. **Status** - Order status (Completed, Pending, Cancelled, Shipped)
11. **PaymentMethod** - Payment type
12. **TotalAmount** - Total order value (Quantity × UnitPrice)
13. **CustomerEmail** - Customer contact email
14. **PhoneNumber** - Customer contact phone
15. **Notes** - Additional comments (sparse)

### 🔍 Data Quality Issues (EASY)

#### Issue #1: Missing Values (6% of data)
- **Location**: ShipDate, DeliveryDate, CustomerEmail columns
- **Type**: NaN/NULL values
- **Real-world scenario**: Incomplete transaction records, pending orders
- **Challenge**: Decide what to do with incomplete records
- **Skills**: Identifying missing data, handling strategies (drop/fill/interpolate)

```
Example: 50 rows have NULL ShipDate (orders not yet shipped)
```

#### Issue #2: Duplicates (3% of data)
- **Location**: Complete row duplicates
- **Type**: Exact duplicate records
- **Real-world scenario**: Double-entry errors, system glitches, API retries
- **Challenge**: Identify and remove duplicates while maintaining data integrity
- **Skills**: Detecting duplicates, handling unique records

```
Example: Order ORD000150 appears twice with identical data
```

#### Issue #3: Inconsistent Date Formats (3% of data)
- **Location**: OrderDate column
- **Type**: Mixed formats (MM/DD/YYYY, DD-MMM-YYYY, datetime objects)
- **Real-world scenario**: Data from multiple sources or system migrations
- **Challenge**: Standardize to single format
- **Skills**: Date parsing, format conversion, datetime handling

```
Before: ['2023-01-15', '01/15/2023', '15-Jan-2023']
After:  ['2023-01-15', '2023-01-15', '2023-01-15']
```

#### Issue #4: Extra Whitespace (4% of data)
- **Location**: ProductName column
- **Type**: Leading/trailing spaces
- **Real-world scenario**: Manual data entry, copy-paste errors
- **Challenge**: Clean whitespace without losing data
- **Skills**: String manipulation, text cleaning

```
Before: 'Laptop  ', '  Mouse', ' Keyboard '
After:  'Laptop', 'Mouse', 'Keyboard'
```

#### Issue #5: Case Inconsistencies (4% of data)
- **Location**: Status column
- **Type**: Mixed case (lowercase, UPPERCASE)
- **Real-world scenario**: Different data entry personnel, legacy systems
- **Challenge**: Standardize categorical values
- **Skills**: Case normalization, categorical standardization

```
Before: ['completed', 'PENDING', 'Cancelled', 'SHIPPED']
After:  ['Completed', 'Pending', 'Cancelled', 'Shipped']
```

### 🎯 Key Cleaning Tasks
1. Identify and handle missing values
2. Remove duplicate rows
3. Standardize date formats
4. Strip and clean whitespace
5. Normalize case in categorical columns
6. Validate data types (ensure numeric columns are numeric)

### 💡 Estimated Time: 1-2 hours

#### **=> Let's follow the Industry order for cleaning**

**STEP 1: Data Ingestion**


In [153]:
import pandas as pd
import numpy as np
import os

# file path.
file_path: str = "messy_datasets_(3 sheets).xlsx"

# confirm the file exists:
if not os.path.exists(path=file_path):
    raise FileNotFoundError(f'{file_path} doesnot exists.')
else:
    try:
        # let's open the whole workbook in format of a dictionary.
        workbook: dict = pd.read_excel(io=file_path, sheet_name=None)

        # Isolating the sheets.
        df_ecommerce: pd.DataFrame = workbook['Easy_Level']
        df_employee: pd.DataFrame = workbook['Medium_Level']
        df_supplychain: pd.DataFrame = workbook['Hard_Level']

        print(f'Succefully opened the workbook contains {len(workbook)} sheets.')
        for ind, name in enumerate(workbook.keys()):
            print(f'{ind + 1} {name}')
    except Exception as e:
        print('Error! {e}')

Succefully opened the workbook contains 3 sheets.
1 Easy_Level
2 Medium_Level
3 Hard_Level


#### **Start cleaning df_ecommerce**

**Step 1:** Data Profiling and Inspection.

In [154]:
# let's seet the dimension of the dataset.
dimension: tuple = df_ecommerce.shape
print(f'The Dataset "df_ecommerce" contains {dimension[0]} rows and {dimension[1]} columns')

# let's see the preassigned datatypes and a qucik review.
print(df_ecommerce.info())

# let's see a statistical overview of some numercial columns in df_ecommerce.
df_ecommerce.describe()

The Dataset "df_ecommerce" contains 1827 rows and 14 columns
<class 'pandas.DataFrame'>
RangeIndex: 1827 entries, 0 to 1826
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   OrderID        1827 non-null   str           
 1   CustomerID     1827 non-null   str           
 2   ProductName    1827 non-null   str           
 3   Category       1827 non-null   str           
 4   Quantity       1827 non-null   int64         
 5   UnitPrice      1827 non-null   float64       
 6   OrderDate      1827 non-null   object        
 7   ShipDate       1790 non-null   datetime64[us]
 8   DeliveryDate   1789 non-null   datetime64[us]
 9   Status         1827 non-null   str           
 10  PaymentMethod  1827 non-null   str           
 11  TotalAmount    1827 non-null   float64       
 12  CustomerEmail  1791 non-null   str           
 13  PhoneNumber    1827 non-null   str           
dtypes: datetime64[us](2), 

,Quantity,UnitPrice,ShipDate,DeliveryDate,TotalAmount
count,1827.000000,1827.000000,1790,1789,1827.000000
mean,9.931582,1023.317163,2023-07-04 03:16:17.430167,2023-07-13 10:02:04.762437,10115.974920
min,1.000000,10.881135,2023-01-01 00:00:00,2023-01-05 00:00:00,52.075231
25%,5.000000,537.236486,2023-04-04 00:00:00,2023-04-16 00:00:00,3061.281484
50%,10.000000,1048.686477,2023-07-05 00:00:00,2023-07-14 00:00:00,7736.537046
75%,15.000000,1525.150361,2023-10-04 00:00:00,2023-10-14 00:00:00,15017.607313
max,19.000000,1999.430470,2024-01-01 00:00:00,2024-01-12 00:00:00,37982.755567
std,5.469844,572.197500,NaN,NaN,8511.199400


In [155]:
# let's check the null value counts.
df_ecommerce.isna().sum()

OrderID           0
CustomerID        0
ProductName       0
Category          0
Quantity          0
UnitPrice         0
OrderDate         0
ShipDate         37
DeliveryDate     38
Status            0
PaymentMethod     0
TotalAmount       0
CustomerEmail    36
PhoneNumber       0
dtype: int64

**Step 2: Structural Standardization**

In [156]:
# The columns are perfectly organized but, still I use the strip for cautions.
df_ecommerce.columns = df_ecommerce.columns.str.strip()

# let's drop if we have a completely empty column fro human readability.
df_ecommerce = df_ecommerce.dropna(axis=1 , how='all')

**Step 3: Deduplication**

In [157]:
# Exact duplicate.
exact_duplicates: int = df_ecommerce.duplicated().sum()
print(f'we have a total of {exact_duplicates} numbers of exact duplicates in "df_ecommerce".')

# let's see them.
df_ecommerce[df_ecommerce.duplicated()]

# just drop the exact duplicates, assgined these to a new dataframe called df_ecomm_clean.

df_ecomm_clean = df_ecommerce.drop_duplicates(ignore_index=True)

print(f'Total number of Exact duplicates in "df_ecomm_clean" is {df_ecomm_clean.duplicated().sum()}')

we have a total of 21 numbers of exact duplicates in "df_ecommerce".
Total number of Exact duplicates in "df_ecomm_clean" is 0


In [158]:
# Let's move on to the partial duplicates.

# Let's check how many duplicates does each columns have.
space = 20
for column in df_ecomm_clean.columns:
    print(f'{column}{" "*(space - len(column))}{df_ecomm_clean[column].duplicated().sum()}')

OrderID             6
CustomerID          1325
ProductName         1782
Category            1803
Quantity            1787
UnitPrice           6
OrderDate           1396
ShipDate            1442
DeliveryDate        1439
Status              1794
PaymentMethod       1802
TotalAmount         6
CustomerEmail       1505
PhoneNumber         6


In [159]:
# since the OrderID should be completely unique, there's no need to keep those
# who posesses the duplicated OrderID.
df_ecomm_clean = df_ecomm_clean.drop_duplicates(subset=['OrderID'], ignore_index=True)

# now let's see a scenario might happen , when a customer accidently "Submit order"
# button twice, for that we will make a composit key , which should not contains
# any duplicates. [CustomerID, ProductName, OrderDate, Quantity, TotalAmount]

# let's check do we have any person fullfilling the following criteria.
df_ecomm_clean.duplicated(subset=["CustomerID", "ProductName", "OrderDate", "Quantity", "TotalAmount"]).sum()
# output is 0 , which means the data set is now ready to move Datatype casting.

np.int64(0)

**Step 3: Data type Casting**

let's see the defualt datatype , assigned by pandas.

In [160]:
# df_ecomm_clean.info()
# DeliveryDate and ShipDate is already a proper datetime object with US formate.
# let's Change the OrderDate column.

# Remove the timing string extension (00:00:00)from every value first, and convert them into datetime64[us] format.
df_ecomm_clean['OrderDate'] = pd.to_datetime(df_ecomm_clean['OrderDate']).dt.date.astype('datetime64[us]')

# let's round our UnitPrice and TotalAmount column into 2 digits.
df_ecomm_clean[['UnitPrice', 'TotalAmount']] = df_ecomm_clean[['UnitPrice', 'TotalAmount']].round(2)
# df_ecomm_clean.info()

**STEP 4: let's Handle the missing value**

In [195]:
# let's check the missing value
print(df_ecomm_clean.isna().sum())

# for handling shiping date missing value let's first sort our df via Orderdate.
df_ecomm_clean = df_ecomm_clean.sort_values(by='OrderDate')

mask = (df_ecomm_clean['ShipDate'].isna()) & (df_ecomm_clean['Status'].isin(['Completed', 'Shipped']))
df_ecomm_clean.loc[mask, 'ShipDate'] = df_ecomm_clean['OrderDate']

# let's create the mask then.
mask_2 = (df_ecomm_clean['DeliveryDate'].notna()) & (df_ecomm_clean['Status'].isin(['Cancelled', 'Pending']))
df_ecomm_clean.loc[mask_2, 'DeliveryDate'] = pd.NaT

df_ecomm_clean[df_ecomm_clean['DeliveryDate'].isna()]

mask_3 = (df_ecomm_clean['DeliveryDate'].isna()) & (df_ecomm_clean['Status']=='Completed')

df_ecomm_clean['DaystoDeliver'] = df_ecomm_clean['DeliveryDate'] - df_ecomm_clean['OrderDate']

delivered = df_ecomm_clean[df_ecomm_clean['DaystoDeliver'].notna()]
averge_day= delivered['DaystoDeliver'].mean()

df_ecomm_clean.loc[mask_3, 'DeliveryDate'] = df_ecomm_clean['OrderDate'] + averge_day.round('D')
df_ecomm_clean = df_ecomm_clean.drop(columns='DaystoDeliver') # Since there's no need to keep DaystoDeliver column.

df_ecomm_clean['CustomerEmail'] = df_ecomm_clean['CustomerEmail'].fillna('Unknown')

# df_ecomm_clean[df_ecomm_clean['CustomerEmail'] == 'Unknown']

OrderID            0
CustomerID         0
ProductName        0
Category           0
Quantity           0
UnitPrice          0
OrderDate          0
ShipDate          17
DeliveryDate     855
Status             0
PaymentMethod      0
TotalAmount        0
CustomerEmail      0
PhoneNumber        0
dtype: int64


In [197]:
df_ecomm_clean.info()

df_ecomm_clean.head()

<class 'pandas.DataFrame'>
Index: 1800 entries, 558 to 210
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   OrderID        1800 non-null   str           
 1   CustomerID     1800 non-null   str           
 2   ProductName    1800 non-null   str           
 3   Category       1800 non-null   str           
 4   Quantity       1800 non-null   int64         
 5   UnitPrice      1800 non-null   float64       
 6   OrderDate      1800 non-null   datetime64[us]
 7   ShipDate       1783 non-null   datetime64[us]
 8   DeliveryDate   945 non-null    datetime64[us]
 9   Status         1800 non-null   str           
 10  PaymentMethod  1800 non-null   str           
 11  TotalAmount    1800 non-null   float64       
 12  CustomerEmail  1800 non-null   str           
 13  PhoneNumber    1800 non-null   str           
dtypes: datetime64[us](3), float64(2), int64(1), str(8)
memory usage: 366.0 KB


,OrderID,CustomerID,ProductName,Category,Quantity,UnitPrice,OrderDate,ShipDate,DeliveryDate,Status,PaymentMethod,TotalAmount,CustomerEmail,PhoneNumber
558,ORD000559,CUST00271,SSD,Accessories,11,1259.71,2023-01-01,2023-01-01,NaT,Cancelled,Credit Card,13856.78,customer258@email.com,+1958-776-8923
179,ORD000180,CUST00398,SSD,Accessories,6,94.16,2023-01-01,2023-01-01,NaT,Pending,Credit Card,564.96,customer179@email.com,+1855-691-9921
521,ORD000522,CUST00013,Mouse,Peripherals,14,630.85,2023-01-01,2023-01-01,2023-01-13,Shipped,Debit Card,8831.87,customer221@email.com,+1785-959-2600
1082,ORD001083,CUST00422,Mouse,Peripherals,18,319.59,2023-01-01,2023-01-01,NaT,Pending,Credit Card,5752.57,customer182@email.com,+1663-898-9870
302,ORD000303,CUST00401,Webcam,Electronics,9,336.12,2023-01-02,2023-01-02,2023-01-17,Completed,Credit Card,3025.11,customer2@email.com,+1738-962-6031


In [199]:
columns_list = ['ProductName', 'Category', 'Status', 'PaymentMethod']
for column in columns_list:
    print(df_ecomm_clean[column].unique())

<ArrowStringArray>
[         'SSD',        'Mouse',       'Webcam',   'Headphones',
          'RAM',      'Charger',     'Keyboard',       'Laptop',
 'Headphones  ',   'Hard Drive',      'Speaker',    'USB Cable',
    'Charger  ',      'Monitor',        'RAM  ',      'Mouse  ',
   'Keyboard  ',    'Monitor  ',     'Laptop  ',    'Speaker  ',
     'Webcam  ',  'USB Cable  ', 'Hard Drive  ',        'SSD  ']
Length: 24, dtype: str
<ArrowStringArray>
['Accessories', 'Peripherals', 'Electronics']
Length: 3, dtype: str
<ArrowStringArray>
['Cancelled',   'Pending',   'Shipped', 'Completed', 'CANCELLED', 'cancelled',
 'COMPLETED',   'PENDING', 'completed',   'SHIPPED',   'shipped',   'pending']
Length: 12, dtype: str
<ArrowStringArray>
['Credit Card', 'Debit Card', 'PayPal', 'Apple Pay']
Length: 4, dtype: str
